# MODULE 7 — Classical Baselines and EEGNet Reference

**Project:** Strict Cross-Dataset, Subject-Independent Motor-Imagery EEG Classification

## Objective

Establish reproducible reference performance before implementing the proposed
domain-generalization architecture.

The same Module 6 cache and fold manifests are used for every baseline.

## Baselines

### B1 — CSP + LDA
One-versus-rest multiclass spatial filtering followed by log-variance features
and Linear Discriminant Analysis.

### B2 — FBCSP + LDA
Five filter-bank bands within the frozen 8–30 Hz representation:

- 8–12 Hz
- 12–16 Hz
- 16–20 Hz
- 20–24 Hz
- 24–30 Hz

CSP is independently fitted within each source-training fold and the resulting
log-variance features are concatenated before LDA.

### B3 — Log-Euclidean covariance + LDA
A compact Riemannian-style covariance baseline:
- shrinkage covariance per epoch;
- matrix logarithm;
- symmetric-matrix vectorization;
- LDA.

The transform is fitted only from source-training epochs.

### B4 — EEGNet reference
A compact EEGNet-style neural baseline using the same 22×640 input.

The EEGNet normalizer is fitted only on source training subjects.

## Evaluation protocols

### Within-dataset LOSO
- BCI-IV-2a: 9 target subjects
- EEGMMIDB: 109 target subjects

### Cross-dataset zero-calibration
- BCI-IV-2a → EEGMMIDB: 109 target subjects
- EEGMMIDB → BCI-IV-2a: 9 target subjects

No target data are used for:
- model fitting;
- feature fitting;
- normalization;
- CSP fitting;
- covariance reference fitting;
- hyperparameter selection.

## Metrics

Primary:
- accuracy
- balanced accuracy
- macro F1

Secondary:
- per-class recall
- confusion matrix
- fold-level mean ± standard deviation

For the 3-class task, chance level is 33.33%.

## Compute policy

The notebook supports:
- fast smoke tests;
- resumable full baseline runs;
- MPS on Apple Silicon when available;
- CPU fallback.

Classical baselines are relatively inexpensive.

EEGNet can be substantially more expensive over 118 LOSO folds, so it is
controlled by explicit configuration flags and writes results after every fold.

## Cell 1 — Imports, configuration, device

In [122]:
# ============================================================
# CELL 1 — IMPORTS + CONFIGURATION (FIXED)
# ============================================================

from __future__ import annotations

import os
import json
import time
import random
import warnings

from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import h5py

from scipy import signal
from scipy.linalg import eigh

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

from sklearn.covariance import LedoitWolf

import mne
from mne.decoding import CSP

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


warnings.filterwarnings("ignore")


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

MANIFEST_ROOT = (
    OUTPUT_ROOT
    / "manifests"
)

CACHE_ROOT = (
    OUTPUT_ROOT
    / "cache"
)

RESULTS_ROOT = (
    OUTPUT_ROOT
    / "results"
)

BASELINE_ROOT = (
    RESULTS_ROOT
    / "module_7_baselines"
)

BASELINE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# REQUIRED INPUT ARTIFACTS
# ============================================================

CACHE_PATH = (
    CACHE_ROOT
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

WITHIN_LOSO_PATH = (
    MANIFEST_ROOT
    / "module_6_within_dataset_loso_folds.csv"
)

TRANSFER_PATH = (
    MANIFEST_ROOT
    / "module_6_cross_dataset_transfer_folds.csv"
)

CACHE_META_PATH = (
    MANIFEST_ROOT
    / "module_6_cache_metadata.csv"
)

PROTOCOL_PATH = (
    MANIFEST_ROOT
    / "module_6_baseline_protocol.json"
)

REQUIRED_ARTIFACTS = [
    CACHE_PATH,
    WITHIN_LOSO_PATH,
    TRANSFER_PATH,
    CACHE_META_PATH,
    PROTOCOL_PATH,
]

for path in REQUIRED_ARTIFACTS:

    assert path.exists(), (
        f"Missing required artifact:\n{path}"
    )


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 20260822

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# DEVICE
# ============================================================

if torch.backends.mps.is_available():

    DEVICE = torch.device(
        "mps"
    )

elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda"
    )

else:

    DEVICE = torch.device(
        "cpu"
    )


# ============================================================
# PRINT ENVIRONMENT
# ============================================================

print("=" * 78)

print(
    "MODULE 7 — BASELINE EXPERIMENTS"
)

print("=" * 78)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device :",
    DEVICE
)

print(
    "Seed   :",
    SEED
)


# ============================================================
# SAFETY CHECK
# ============================================================

print(
    "\nRequired artifacts:"
)

for path in REQUIRED_ARTIFACTS:

    print(
        "  ✓",
        path
    )

print(
    "\nCell 1 imports/configuration: PASS"
)

MODULE 7 — BASELINE EXPERIMENTS
PyTorch: 2.10.0
Device : mps
Seed   : 20260822

Required artifacts:
  ✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5
  ✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_within_dataset_loso_folds.csv
  ✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cross_dataset_transfer_folds.csv
  ✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cache_metadata.csv
  ✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_baseline_protocol.json

Cell 1 imports/configuration: PASS


## Cell 2 — Frozen experiment configuration

In [123]:
# ============================================================
# CELL 2 — FROZEN BASELINE CONFIGURATION (FINAL)
# ============================================================

PRIMARY_CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    "left": 0,
    "right": 1,
    "feet": 2,
}

ID_TO_CLASS = {
    v: k
    for k, v in CLASS_TO_ID.items()
}


# ============================================================
# CSP
# ============================================================

CSP_N_COMPONENTS = 6


# ============================================================
# FBCSP
# ============================================================

FBCSP_BANDS = [
    (8.0, 12.0),
    (12.0, 16.0),
    (16.0, 20.0),
    (20.0, 24.0),
    (24.0, 30.0),
]

FBCSP_N_COMPONENTS = 4


# ============================================================
# RIEMANNIAN
# ============================================================

RIEMANNIAN_REG = "oas"


# ============================================================
# EEGNET
# ============================================================

EEGNET_EPOCHS = 30
EEGNET_BATCH_SIZE = 128
EEGNET_LR = 1e-3
EEGNET_WEIGHT_DECAY = 1e-4
EEGNET_PATIENCE = 6


# ============================================================
# BASELINE SWITCHES
# ============================================================

RUN_CSP = True
RUN_FBCSP = True
RUN_RIEMANNIAN = True
RUN_EEGNET = True


# ============================================================
# EXPERIMENT MODE
# ============================================================

# ------------------------------------------------------------
# FINAL FULL EXPERIMENT
# ------------------------------------------------------------

SMOKE_TEST = False

# Only used when SMOKE_TEST=True.
SMOKE_FOLDS_PER_PROTOCOL = 1


# ------------------------------------------------------------
# EEGNet FULL-RUN CONTROL
#
# IMPORTANT:
# Leave these False initially so the classical baselines
# can complete first.
#
# After CSP/FBCSP/Riemannian have completed and their results
# are verified, set these to True for the complete EEGNet
# evaluation.
# ------------------------------------------------------------

EEGNET_FULL_WITHIN_LOSO = False
EEGNET_FULL_CROSS_DATASET = False


# ============================================================
# RESULT CHECKPOINTING
# ============================================================

RESULTS_FLUSH_EVERY_FOLD = True


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("=" * 78)

print(
    "MODULE 7 CONFIGURATION"
)

print("=" * 78)

print(
    json.dumps(
        {
            "classes": PRIMARY_CLASSES,

            "csp_components":
                CSP_N_COMPONENTS,

            "fbcsp_bands":
                FBCSP_BANDS,

            "fbcsp_components":
                FBCSP_N_COMPONENTS,

            "riemannian_regularization":
                RIEMANNIAN_REG,

            "eegnet_epochs":
                EEGNET_EPOCHS,

            "eegnet_batch_size":
                EEGNET_BATCH_SIZE,

            "run_csp":
                RUN_CSP,

            "run_fbcsp":
                RUN_FBCSP,

            "run_riemannian":
                RUN_RIEMANNIAN,

            "run_eegnet":
                RUN_EEGNET,

            "smoke_test":
                SMOKE_TEST,

            "smoke_folds_per_protocol":
                SMOKE_FOLDS_PER_PROTOCOL,

            "eegnet_full_within_loso":
                EEGNET_FULL_WITHIN_LOSO,

            "eegnet_full_cross_dataset":
                EEGNET_FULL_CROSS_DATASET,
        },
        indent=2,
    )
)


# ============================================================
# MODE VALIDATION
# ============================================================

if SMOKE_TEST:

    print(
        "\nMODE: SMOKE TEST"
    )

    print(
        "Only the configured smoke folds will run."
    )

else:

    print(
        "\nMODE: FULL EXPERIMENT"
    )

    print(
        "Complete classical baseline evaluation is enabled."
    )

    print(
        "EEGNet full LOSO:",
        EEGNET_FULL_WITHIN_LOSO,
    )

    print(
        "EEGNet full cross-dataset:",
        EEGNET_FULL_CROSS_DATASET,
    )


print(
    "\nCell 2 configuration validation: PASS"
)

MODULE 7 CONFIGURATION
{
  "classes": [
    "left",
    "right",
    "feet"
  ],
  "csp_components": 6,
  "fbcsp_bands": [
    [
      8.0,
      12.0
    ],
    [
      12.0,
      16.0
    ],
    [
      16.0,
      20.0
    ],
    [
      20.0,
      24.0
    ],
    [
      24.0,
      30.0
    ]
  ],
  "fbcsp_components": 4,
  "riemannian_regularization": "oas",
  "eegnet_epochs": 30,
  "eegnet_batch_size": 128,
  "run_csp": true,
  "run_fbcsp": true,
  "run_riemannian": true,
  "run_eegnet": true,
  "smoke_test": false,
  "smoke_folds_per_protocol": 1,
  "eegnet_full_within_loso": false,
  "eegnet_full_cross_dataset": false
}

MODE: FULL EXPERIMENT
Complete classical baseline evaluation is enabled.
EEGNet full LOSO: False
EEGNet full cross-dataset: False

Cell 2 configuration validation: PASS


## Cell 3 — Load cache metadata and fold manifests

In [124]:
# ============================================================
# CELL 3 — LOAD METADATA + FOLD MANIFESTS
# ============================================================

cache_meta_df = pd.read_csv(
    CACHE_META_PATH
)

within_loso_df = pd.read_csv(
    WITHIN_LOSO_PATH
)

transfer_df = pd.read_csv(
    TRANSFER_PATH
)

print("Cache epochs :", len(cache_meta_df))
print(
    "Cache subjects:",
    cache_meta_df["subject"].nunique(),
)
print(
    "Within LOSO folds:",
    len(within_loso_df),
)
print(
    "Transfer folds:",
    len(transfer_df),
)

assert len(cache_meta_df) == 9316
assert cache_meta_df["subject"].nunique() == 118

assert len(within_loso_df) == 118
assert len(transfer_df) == 118

print("\nModule 6 artifacts loaded: PASS")

Cache epochs : 9316
Cache subjects: 118
Within LOSO folds: 118
Transfer folds: 118

Module 6 artifacts loaded: PASS


## Cell 4 — HDF5 batch reader

In [125]:
# ============================================================
# CELL 4 — HDF5 BATCH READER
# ============================================================

class HDF5Store:
    def __init__(self, path):
        self.path = Path(path)
        self.h5 = None

    def __enter__(self):
        self.h5 = h5py.File(
            self.path,
            "r",
        )
        return self

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        if self.h5 is not None:
            self.h5.close()
            self.h5 = None

    def get_X(
        self,
        indices,
    ):
        indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        if len(indices) == 0:
            return np.empty(
                (0, 22, 640),
                dtype=np.float32,
            )

        order = np.argsort(
            indices
        )

        sorted_idx = indices[
            order
        ]

        X_sorted = np.asarray(
            self.h5["X"][
                sorted_idx
            ],
            dtype=np.float32,
        )

        inverse = np.argsort(
            order
        )

        return X_sorted[
            inverse
        ]

    def get_meta(
        self,
        indices,
    ):
        indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        order = np.argsort(
            indices
        )

        sorted_idx = indices[
            order
        ]

        inverse = np.argsort(
            order
        )

        result = {}

        string_keys = [
            "dataset",
            "subject",
            "run",
            "recording_id",
            "harmonized_class",
        ]

        for key in string_keys:
            vals = self.h5[
                "metadata"
            ][key][
                sorted_idx
            ]

            result[key] = [
                x.decode("utf-8")
                if isinstance(x, bytes)
                else str(x)
                for x in vals
            ]

            result[key] = [
                result[key][i]
                for i in inverse
            ]

        result["event_index"] = (
            self.h5[
                "metadata"
            ]["event_index"][
                sorted_idx
            ][inverse]
        )

        result["source_sfreq_hz"] = (
            self.h5[
                "metadata"
            ]["source_sfreq_hz"][
                sorted_idx
            ][inverse]
        )

        return pd.DataFrame(
            result,
            index=indices,
        )


print("HDF5 reader ready.")

HDF5 reader ready.


## Cell 5 — Metrics and result schema

In [126]:
# ============================================================
# CELL 5 — METRICS
# ============================================================

def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int64,
    )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
            )
        ),
        "recall_left": float(
            f1_score(
                y_true,
                y_pred,
                labels=[0],
                average=None,
                zero_division=0,
            )[0]
        ),
        "recall_right": float(
            f1_score(
                y_true,
                y_pred,
                labels=[1],
                average=None,
                zero_division=0,
            )[0]
        ),
        "recall_feet": float(
            f1_score(
                y_true,
                y_pred,
                labels=[2],
                average=None,
                zero_division=0,
            )[0]
        ),
    }


def safe_protocol_indices(
    json_string,
):
    return np.asarray(
        json.loads(json_string),
        dtype=np.int64,
    )


def protocol_name_from_row(row):
    return str(
        row["protocol"]
    )


print(
    "Metrics and fold-index utilities: PASS"
)

Metrics and fold-index utilities: PASS


## Cell 6 — Source-only robust normalizer

In [127]:
# ============================================================
# CELL 6 — SOURCE-ONLY NORMALIZATION
# ============================================================

class FoldRobustNormalizer:
    """
    Fold-local channel-wise robust normalizer.

    Fits only on the source-training partition.
    """

    def __init__(
        self,
        eps=1e-6,
    ):
        self.eps = float(eps)
        self.median_ = None
        self.iqr_ = None
        self.fit_subjects_ = None

    def fit(
        self,
        X,
        subjects,
    ):

        X = np.asarray(
            X,
            dtype=np.float64,
        )

        subjects = [
            str(s)
            for s in subjects
        ]

        if X.ndim != 3:
            raise ValueError(X.shape)

        if len(subjects) != X.shape[0]:
            raise ValueError(
                "Subject count must match X epochs."
            )

        values = (
            X.transpose(
                1,
                0,
                2,
            )
            .reshape(
                22,
                -1,
            )
        )

        self.median_ = np.median(
            values,
            axis=1,
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        self.fit_subjects_ = tuple(
            sorted(
                set(subjects)
            )
        )

        return self

    def transform(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float64,
        )

        return (
            X
            - self.median_[
                None,
                :,
                None,
            ]
        ) / (
            self.iqr_[
                None,
                :,
                None,
            ]
        )

    def assert_target_excluded(
        self,
        target_subject,
    ):

        target_subject = str(
            target_subject
        )

        assert (
            target_subject
            not in set(
                self.fit_subjects_
            )
        ), (
            f"LEAKAGE: {target_subject} "
            "was used in normalizer fit."
        )


print(
    "Fold-local normalization utility: PASS"
)

Fold-local normalization utility: PASS


## Cell 7 — CSP + LDA implementation

CSP is fitted independently inside each fold.

The target subject is never involved in CSP fitting.

MNE's multiclass CSP generates a spatial representation followed by LDA.

In [128]:
# ============================================================
# CELL 7 — CSP + LDA
# ============================================================

def fit_csp_lda(
    X_train,
    y_train,
    n_components=CSP_N_COMPONENTS,
):

    csp = CSP(
        n_components=n_components,
        reg="oas",
        log=True,
        norm_trace=False,
        transform_into="average_power",
    )

    X_train_features = csp.fit_transform(
        X_train,
        y_train,
    )

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
    )

    lda.fit(
        X_train_features,
        y_train,
    )

    return csp, lda


def predict_csp_lda(
    csp,
    lda,
    X_test,
):

    X_features = csp.transform(
        X_test
    )

    return lda.predict(
        X_features
    )


print("CSP + LDA implementation: READY")

CSP + LDA implementation: READY


## Cell 8 — FBCSP + LDA implementation

The 8–30 Hz cache is passed through a deterministic filter bank.

Each band gets:
- band-pass filter;
- CSP fit on source training data;
- log-power features.

Only source training data determine CSP spatial filters and LDA parameters.

## FBCSP-specific broadband representation

The primary Module 5 cache is already 8–30 Hz and therefore is not a valid
starting point for a full filter-bank CSP baseline. Module 7 creates a
separate deterministic **4–45 Hz broadband cache** using the original
recordings, frozen 22-channel montage, continuous average reference,
continuous resampling to 160 Hz, and continuous 4–45 Hz filtering.

The FBCSP bands are then extracted from this broadband representation.

In [129]:
# ============================================================
# CELL 8A — BUILD FBCSP BROADBAND CACHE
#              SMOKE-TEST / FULL-RUN SAFE
# ============================================================

from tqdm.auto import tqdm

FBCSP_BROADBAND_LOW = 4.0
FBCSP_BROADBAND_HIGH = 45.0
FBCSP_BROADBAND_SFREQ = 160.0

FBCSP_BROADBAND_CACHE = (
    BASELINE_ROOT
    / "fbcsp_broadband_4_45hz_160hz.h5"
)

FBCSP_BROADBAND_META = (
    BASELINE_ROOT
    / "fbcsp_broadband_metadata.csv"
)


# ============================================================
# LOAD MODULE 4 FROZEN CHANNEL MAP
# ============================================================

CHANNEL_MAP_PATH = (
    MANIFEST_ROOT
    / "module_4_common_channel_mapping.csv"
)

FROZEN_CHANNELS_PATH = (
    MANIFEST_ROOT
    / "module_4_frozen_common_channels.json"
)

assert CHANNEL_MAP_PATH.exists()
assert FROZEN_CHANNELS_PATH.exists()

channel_map_df = pd.read_csv(
    CHANNEL_MAP_PATH
)

with open(
    FROZEN_CHANNELS_PATH,
    "r",
    encoding="utf-8",
) as f:

    frozen_spec = json.load(f)

FROZEN_CHANNELS = list(
    frozen_spec["channels"]
)

assert len(
    FROZEN_CHANNELS
) == 22


bci_map = {
    row["canonical_electrode"]:
        int(
            row["bci_channel_index"]
        )
    for _, row in channel_map_df.iterrows()
}

phys_map = {
    row["canonical_electrode"]:
        int(
            row["physionet_channel_index"]
        )
    for _, row in channel_map_df.iterrows()
}


# ============================================================
# RAW LOADER
# ============================================================

def open_fbcsp_raw(
    path,
    dataset,
):

    if dataset == "BCI-IV-2a":

        return mne.io.read_raw_gdf(
            str(path),
            preload=True,
            verbose="ERROR",
        )

    if dataset == "EEGMMIDB":

        return mne.io.read_raw_edf(
            str(path),
            preload=True,
            verbose="ERROR",
        )

    raise ValueError(
        f"Unsupported dataset: {dataset}"
    )


# ============================================================
# 22-CHANNEL EXTRACTION
# ============================================================

def extract_fbcsp_22(
    raw,
    dataset,
):

    mapping = (
        bci_map
        if dataset == "BCI-IV-2a"
        else phys_map
    )

    indices = [
        mapping[ch]
        for ch in FROZEN_CHANNELS
    ]

    X = raw.get_data(
        picks=indices
    ).astype(
        np.float64
    )

    assert X.shape[0] == 22

    return X


# ============================================================
# CONTINUOUS RESAMPLING
# ============================================================

def resample_fbcsp_continuous(
    X,
    source_sfreq,
):

    if np.isclose(
        source_sfreq,
        FBCSP_BROADBAND_SFREQ,
    ):

        return X.copy()

    from fractions import Fraction

    ratio = Fraction(
        FBCSP_BROADBAND_SFREQ
        / source_sfreq
    ).limit_denominator(
        1000
    )

    return signal.resample_poly(
        X,
        up=ratio.numerator,
        down=ratio.denominator,
        axis=-1,
        padtype="line",
    ).astype(
        np.float64
    )


# ============================================================
# CONTINUOUS 4–45 Hz FILTER
# ============================================================

def filter_fbcsp_continuous(
    X,
    sfreq,
):

    info = mne.create_info(
        ch_names=[
            f"EEG{i:02d}"
            for i in range(22)
        ],
        sfreq=float(sfreq),
        ch_types=[
            "eeg"
        ] * 22,
    )

    temp_raw = mne.io.RawArray(
        X,
        info,
        verbose="ERROR",
    )

    try:

        temp_raw.filter(
            l_freq=FBCSP_BROADBAND_LOW,
            h_freq=FBCSP_BROADBAND_HIGH,
            method="fir",
            phase="zero",
            fir_design="firwin",
            verbose="ERROR",
        )

        return temp_raw.get_data().astype(
            np.float64
        )

    finally:

        temp_raw.close()


# ============================================================
# FINAL CACHE METADATA
# ============================================================

fbcsp_source_meta = (
    cache_meta_df[
        [
            "cache_index",
            "dataset",
            "subject",
            "run",
            "recording_id",
            "absolute_path",
            "event_index",
            "onset_sec",
            "harmonized_class",
            "source_sfreq_hz",
        ]
    ]
    .sort_values(
        "cache_index"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# DETERMINE EXACT EPOCHS NEEDED
# ============================================================

if SMOKE_TEST:

    print("=" * 78)
    print("FBCSP BROADBAND CACHE — TRUE SMOKE MODE")
    print("=" * 78)

    # --------------------------------------------------------
    # First fold from each dataset for within-dataset LOSO
    # --------------------------------------------------------

    smoke_within = (
        within_loso_df
        .groupby(
            "dataset",
            group_keys=False,
        )
        .head(
            SMOKE_FOLDS_PER_PROTOCOL
        )
        .copy()
    )

    # --------------------------------------------------------
    # First fold from each cross-dataset direction
    # --------------------------------------------------------

    smoke_transfer = (
        transfer_df
        .groupby(
            [
                "source_dataset",
                "target_dataset",
            ],
            group_keys=False,
        )
        .head(
            SMOKE_FOLDS_PER_PROTOCOL
        )
        .copy()
    )

    required_indices = set()

    # Within-dataset folds
    for _, fold in smoke_within.iterrows():

        required_indices.update(
            json.loads(
                fold["train_indices_json"]
            )
        )

        required_indices.update(
            json.loads(
                fold["test_indices_json"]
            )
        )

    # Cross-dataset folds
    for _, fold in smoke_transfer.iterrows():

        required_indices.update(
            json.loads(
                fold["train_indices_json"]
            )
        )

        required_indices.update(
            json.loads(
                fold["test_indices_json"]
            )
        )

    required_indices = np.asarray(
        sorted(
            required_indices
        ),
        dtype=np.int64,
    )

    selected_meta = (
        fbcsp_source_meta[
            fbcsp_source_meta[
                "cache_index"
            ].isin(
                required_indices
            )
        ]
        .copy()
    )

    print(
        "Within smoke folds:",
        len(smoke_within)
    )

    print(
        "Transfer smoke folds:",
        len(smoke_transfer)
    )

    print(
        "Required FBCSP epochs:",
        len(selected_meta)
    )

else:

    # Full experiment
    selected_meta = (
        fbcsp_source_meta.copy()
    )

    print("=" * 78)
    print("FBCSP BROADBAND CACHE — FULL MODE")
    print("=" * 78)

    print(
        "Required FBCSP epochs:",
        len(selected_meta)
    )


# ============================================================
# REMOVE PREVIOUS CACHE
# ============================================================

if FBCSP_BROADBAND_CACHE.exists():

    FBCSP_BROADBAND_CACHE.unlink()


# ============================================================
# PREPARE OUTPUT
# ============================================================

epoch_arrays = []
epoch_rows = []

recording_groups = (
    selected_meta
    .groupby(
        [
            "dataset",
            "subject",
            "recording_id",
        ],
        sort=False,
    )
)


print(
    "Required recordings:",
    selected_meta[
        "recording_id"
    ].nunique()
)


# ============================================================
# PROCESS CONTINUOUS RECORDINGS
# ============================================================

for _, events in tqdm(
    recording_groups,
    total=selected_meta[
        "recording_id"
    ].nunique(),
    desc="Building FBCSP broadband cache",
):

    first = events.iloc[0]

    raw = open_fbcsp_raw(
        first["absolute_path"],
        first["dataset"],
    )

    try:

        source_sfreq = float(
            raw.info["sfreq"]
        )

        X_cont = extract_fbcsp_22(
            raw,
            first["dataset"],
        )

    finally:

        try:
            raw.close()
        except Exception:
            pass


    # --------------------------------------------------------
    # Continuous average reference
    # --------------------------------------------------------

    X_cont = (
        X_cont
        - X_cont.mean(
            axis=0,
            keepdims=True,
        )
    )


    # --------------------------------------------------------
    # Continuous resampling
    # --------------------------------------------------------

    X_cont = (
        resample_fbcsp_continuous(
            X_cont,
            source_sfreq,
        )
    )


    # --------------------------------------------------------
    # Continuous 4–45 Hz broadband filtering
    # --------------------------------------------------------

    X_cont = (
        filter_fbcsp_continuous(
            X_cont,
            FBCSP_BROADBAND_SFREQ,
        )
    )


    # --------------------------------------------------------
    # Extract required events only
    # --------------------------------------------------------

    for _, event in events.iterrows():

        start = int(
            round(
                float(
                    event["onset_sec"]
                )
                * FBCSP_BROADBAND_SFREQ
            )
        )

        stop = (
            start
            + 640
        )

        if (
            start < 0
            or stop > X_cont.shape[1]
        ):

            raise RuntimeError(
                "FBCSP broadband epoch unavailable: "
                f"cache_index="
                f"{event['cache_index']}"
            )

        epoch = (
            X_cont[
                :,
                start:stop,
            ]
            .astype(
                np.float32
            )
        )

        assert epoch.shape == (
            22,
            640,
        )

        assert np.isfinite(
            epoch
        ).all()

        epoch_arrays.append(
            epoch
        )

        epoch_rows.append(
            event.to_dict()
        )


# ============================================================
# STACK
# ============================================================

X_broadband = np.stack(
    epoch_arrays,
    axis=0,
)


fbcsp_broadband_meta_df = (
    pd.DataFrame(
        epoch_rows
    )
    .sort_values(
        "cache_index"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# CRITICAL INDEX ALIGNMENT
# ============================================================

assert (
    fbcsp_broadband_meta_df[
        "cache_index"
    ]
    .is_unique
)

assert np.array_equal(
    fbcsp_broadband_meta_df[
        "cache_index"
    ].to_numpy(),
    selected_meta[
        "cache_index"
    ].to_numpy(),
)


assert X_broadband.shape == (
    len(
        fbcsp_broadband_meta_df
    ),
    22,
    640,
)


# ============================================================
# WRITE CACHE
# ============================================================

with h5py.File(
    FBCSP_BROADBAND_CACHE,
    "w",
) as h5:

    h5.create_dataset(
        "X_broadband",
        data=X_broadband,
        dtype=np.float32,
        compression="gzip",
        compression_opts=4,
    )

    h5.attrs[
        "module"
    ] = 7

    h5.attrs[
        "representation"
    ] = "FBCSP_broadband_4_45Hz"

    h5.attrs[
        "low_hz"
    ] = FBCSP_BROADBAND_LOW

    h5.attrs[
        "high_hz"
    ] = FBCSP_BROADBAND_HIGH

    h5.attrs[
        "sfreq_hz"
    ] = FBCSP_BROADBAND_SFREQ

    h5.attrs[
        "n_channels"
    ] = 22

    h5.attrs[
        "n_samples"
    ] = 640

    h5.attrs[
        "continuous_preprocessing"
    ] = True

    h5.attrs[
        "normalized"
    ] = False

    h5.attrs[
        "smoke_test"
    ] = bool(
        SMOKE_TEST
    )


# ============================================================
# SAVE METADATA
# ============================================================

fbcsp_broadband_meta_df.to_csv(
    FBCSP_BROADBAND_META,
    index=False,
)


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 78
)

print(
    "FBCSP BROADBAND CACHE COMPLETE"
)

print(
    "=" * 78
)

print(
    "Mode:",
    "SMOKE TEST"
    if SMOKE_TEST
    else "FULL EXPERIMENT"
)

print(
    "Epochs:",
    len(
        fbcsp_broadband_meta_df
    )
)

print(
    "Shape:",
    X_broadband.shape
)

print(
    "Source rates used:"
)

display(
    fbcsp_broadband_meta_df[
        [
            "dataset",
            "source_sfreq_hz",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset",
            "source_sfreq_hz",
        ]
    )
)

print(
    "\nRepresentation:",
    "continuous 4–45 Hz"
)

print(
    "Target sampling rate:",
    "160 Hz"
)

print(
    "Channels:",
    22
)

print(
    "Samples:",
    640
)

print(
    "\nFBCSP broadband cache validation: PASS"
)

FBCSP BROADBAND CACHE — FULL MODE
Required FBCSP epochs: 9316
Required recordings: 663


Building FBCSP broadband cache:   0%|          | 0/663 [00:00<?, ?it/s]


FBCSP BROADBAND CACHE COMPLETE
Mode: FULL EXPERIMENT
Epochs: 9316
Shape: (9316, 22, 640)
Source rates used:


,dataset,source_sfreq_hz
0,BCI-IV-2a,250.0
7812,EEGMMIDB,128.0
1944,EEGMMIDB,160.0



Representation: continuous 4–45 Hz
Target sampling rate: 160 Hz
Channels: 22
Samples: 640

FBCSP broadband cache validation: PASS


In [130]:
# ============================================================
# FBCSP BROADBAND LOADER — GLOBAL INDEX SAFE
# ============================================================

# Build a mapping from the original Module 6 cache index
# to the local row of the FBCSP broadband cache.
#
# This works correctly for BOTH:
#   SMOKE_TEST=True
#   SMOKE_TEST=False
#
# because the FBCSP metadata preserves the original
# Module 6 cache_index.

assert FBCSP_BROADBAND_META.exists(), (
    f"Missing FBCSP broadband metadata:\n"
    f"{FBCSP_BROADBAND_META}"
)

fbcsp_index_map_df = pd.read_csv(
    FBCSP_BROADBAND_META
)

assert (
    "cache_index"
    in fbcsp_index_map_df.columns
)

assert (
    fbcsp_index_map_df[
        "cache_index"
    ].is_unique
), (
    "FBCSP broadband metadata contains "
    "duplicate global cache indices."
)

FBCSP_GLOBAL_TO_LOCAL = {
    int(global_idx): int(local_idx)
    for local_idx, global_idx
    in enumerate(
        fbcsp_index_map_df[
            "cache_index"
        ].to_numpy()
    )
}


def load_fbcsp_broadband(
    indices,
):
    """
    Load FBCSP broadband epochs using the ORIGINAL
    Module 6 cache indices.

    Works in both:
        SMOKE_TEST=True
        SMOKE_TEST=False
    """

    global_indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(global_indices) == 0:
        return np.empty(
            (
                0,
                22,
                640,
            ),
            dtype=np.float32,
        )

    missing = [
        int(idx)
        for idx in global_indices
        if int(idx)
        not in FBCSP_GLOBAL_TO_LOCAL
    ]

    if missing:

        raise KeyError(
            "The following Module 6 cache indices "
            "are not present in the FBCSP broadband cache: "
            f"{missing[:20]}"
            + (
                " ..."
                if len(missing) > 20
                else ""
            )
        )

    # Global Module-6 index -> local FBCSP row
    local_indices = np.asarray(
        [
            FBCSP_GLOBAL_TO_LOCAL[
                int(idx)
            ]
            for idx in global_indices
        ],
        dtype=np.int64,
    )

    with h5py.File(
        FBCSP_BROADBAND_CACHE,
        "r",
    ) as h5:

        order = np.argsort(
            local_indices
        )

        sorted_local = (
            local_indices[order]
        )

        X_sorted = np.asarray(
            h5[
                "X_broadband"
            ][sorted_local],
            dtype=np.float32,
        )

        inverse = np.argsort(
            order
        )

        X = X_sorted[
            inverse
        ]

    assert X.shape == (
        len(global_indices),
        22,
        640,
    )

    assert np.isfinite(
        X
    ).all()

    return X

## Cell 9 — Log-Euclidean covariance + LDA

This baseline avoids an external pyRiemann dependency.

For every epoch:

1. shrinkage covariance using Ledoit-Wolf;
2. matrix logarithm via eigen-decomposition;
3. vectorize the upper triangle.

The transform is learned from the source training data only through the LDA
classifier; covariance construction itself is per-epoch and label-free.

In [131]:
# ============================================================
# FBCSP GLOBAL → LOCAL INDEX VERIFICATION
# ============================================================

print("=" * 78)
print("FBCSP GLOBAL/LOCAL INDEX VERIFICATION")
print("=" * 78)

test_global_indices = (
    fbcsp_index_map_df[
        "cache_index"
    ]
    .head(
        min(
            5,
            len(fbcsp_index_map_df),
        )
    )
    .to_numpy(
        dtype=np.int64,
    )
)

test_local_indices = np.asarray(
    [
        FBCSP_GLOBAL_TO_LOCAL[
            int(idx)
        ]
        for idx in test_global_indices
    ],
    dtype=np.int64,
)

print(
    "Global cache indices:",
    test_global_indices.tolist(),
)

print(
    "Mapped local rows:",
    test_local_indices.tolist(),
)

assert np.all(
    test_local_indices
    >= 0
)

assert np.all(
    test_local_indices
    < len(
        fbcsp_index_map_df
    )
)

X_index_test = (
    load_fbcsp_broadband(
        test_global_indices
    )
)

print(
    "Retrieved shape:",
    X_index_test.shape
)

assert X_index_test.shape == (
    len(test_global_indices),
    22,
    640,
)

print(
    "\nGlobal/local FBCSP indexing: PASS"
)

FBCSP GLOBAL/LOCAL INDEX VERIFICATION
Global cache indices: [0, 1, 2, 3, 4]
Mapped local rows: [0, 1, 2, 3, 4]
Retrieved shape: (5, 22, 640)

Global/local FBCSP indexing: PASS


In [132]:
# ============================================================
# CELL 9 — LOG-EUCLIDEAN COVARIANCE + LDA
# ============================================================

def covariance_log_vector(
    epoch,
):
    """
    Log-Euclidean SPD covariance representation.
    """

    X = np.asarray(
        epoch,
        dtype=np.float64,
    )

    # Center each channel.
    X = (
        X
        - X.mean(
            axis=1,
            keepdims=True,
        )
    )

    # Ledoit-Wolf shrinkage covariance.
    lw = LedoitWolf(
        assume_centered=True
    )

    cov = lw.fit(
        X.T
    ).covariance_

    cov = (
        cov
        + cov.T
    ) / 2.0

    eigvals, eigvecs = eigh(
        cov
    )

    eigvals = np.maximum(
        eigvals,
        1e-10,
    )

    log_cov = (
        eigvecs
        @ np.diag(
            np.log(
                eigvals
            )
        )
        @ eigvecs.T
    )

    # Upper triangular vectorization.
    idx = np.triu_indices(
        log_cov.shape[0]
    )

    return log_cov[
        idx
    ]


def riemannian_features(
    X,
):

    return np.stack(
        [
            covariance_log_vector(
                epoch
            )
            for epoch in X
        ]
    )


def fit_riemannian_lda(
    X_train,
    y_train,
):

    train_features = (
        riemannian_features(
            X_train
        )
    )

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
    )

    lda.fit(
        train_features,
        y_train,
    )

    return lda


def predict_riemannian_lda(
    lda,
    X_test,
):

    features = (
        riemannian_features(
            X_test
        )
    )

    return lda.predict(
        features
    )


print(
    "Log-Euclidean covariance + LDA: READY"
)

Log-Euclidean covariance + LDA: READY


## Cell 10 — EEGNet reference architecture

This is a compact EEGNet-style reference, not the proposed domain-generalized
architecture.

Input:
`(B, 1, 22, 640)`

The network is deliberately small enough for MacBook M4 MPS/CPU execution.

In [133]:
# ============================================================
# CELL 10 — EEGNET REFERENCE
# ============================================================

class EEGNetReference(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_times=640,
        n_classes=3,
        dropout=0.25,
    ):

        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(
                1,
                8,
                kernel_size=(
                    1,
                    64,
                ),
                padding=(
                    0,
                    32,
                ),
                bias=False,
            ),
            nn.BatchNorm2d(
                8
            ),
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(
                8,
                16,
                kernel_size=(
                    n_channels,
                    1,
                ),
                groups=8,
                bias=False,
            ),
            nn.BatchNorm2d(
                16
            ),
            nn.ELU(),
            nn.AvgPool2d(
                kernel_size=(
                    1,
                    4,
                )
            ),
            nn.Dropout(
                dropout
            ),
        )

        self.separable = nn.Sequential(
            nn.Conv2d(
                16,
                16,
                kernel_size=(
                    1,
                    16,
                ),
                padding=(
                    0,
                    8,
                ),
                groups=16,
                bias=False,
            ),
            nn.Conv2d(
                16,
                16,
                kernel_size=(
                    1,
                    1,
                ),
                bias=False,
            ),
            nn.BatchNorm2d(
                16
            ),
            nn.ELU(),
            nn.AvgPool2d(
                kernel_size=(
                    1,
                    8,
                )
            ),
            nn.Dropout(
                dropout
            ),
        )

        with torch.no_grad():
            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_times,
            )

            shape = (
                self.separable(
                    self.depthwise(
                        self.temporal(
                            dummy
                        )
                    )
                )
                .shape
            )

        self.classifier = nn.Linear(
            int(
                np.prod(
                    shape[1:]
                )
            ),
            n_classes,
        )

    def forward(
        self,
        x,
    ):

        # x shape:
        # (B,22,640)
        if x.ndim == 3:
            x = x.unsqueeze(1)

        x = self.temporal(
            x
        )

        x = self.depthwise(
            x
        )

        x = self.separable(
            x
        )

        x = x.flatten(
            start_dim=1
        )

        return self.classifier(
            x
        )


print(
    "EEGNet reference parameters:"
)

test_model = EEGNetReference()

print(
    sum(
        p.numel()
        for p in test_model.parameters()
    )
)

print(
    "EEGNet architecture: PASS"
)

EEGNet reference parameters:
2419
EEGNet architecture: PASS


## Cell 11 — EEGNet training utility

Training uses only source training data.

Validation is created from source subjects only and is used for early stopping.
The target subject is never used for model selection.

In [134]:
# ============================================================
# CELL 11 — EEGNET TRAINING
# ============================================================

def train_eegnet(
    X_train,
    y_train,
    source_subjects,
    seed=SEED,
):

    torch.manual_seed(
        seed
    )

    if DEVICE.type == "mps":
        torch.mps.manual_seed(
            seed
        )

    # --------------------------------------------------------
    # Deterministic source-only validation split by subject.
    # Use the final source subject as validation subject.
    # This keeps validation subject-independent.
    # --------------------------------------------------------

    unique_subjects = sorted(
        set(
            str(s)
            for s in source_subjects
        )
    )

    if len(unique_subjects) < 2:
        raise ValueError(
            "EEGNet requires at least two source subjects "
            "for a subject-level validation split."
        )

    val_subject = unique_subjects[
        -1
    ]

    train_mask = np.array([
        str(s) != val_subject
        for s in source_subjects
    ])

    val_mask = ~train_mask

    X_fit = X_train[
        train_mask
    ]

    y_fit = y_train[
        train_mask
    ]

    X_val = X_train[
        val_mask
    ]

    y_val = y_train[
        val_mask
    ]

    fit_subjects = [
        str(s)
        for i, s in enumerate(
            source_subjects
        )
        if train_mask[i]
    ]

    # --------------------------------------------------------
    # Fit normalization only on training subjects.
    # --------------------------------------------------------

    normalizer = (
        FoldRobustNormalizer()
        .fit(
            X_fit,
            fit_subjects,
        )
    )

    normalizer.assert_target_excluded(
        val_subject
    )

    X_fit = normalizer.transform(
        X_fit
    ).astype(
        np.float32
    )

    X_val = normalizer.transform(
        X_val
    ).astype(
        np.float32
    )

    # --------------------------------------------------------
    # Torch datasets
    # --------------------------------------------------------

    train_ds = TensorDataset(
        torch.from_numpy(
            X_fit
        ),
        torch.from_numpy(
            y_fit.astype(
                np.int64
            )
        ),
    )

    val_ds = TensorDataset(
        torch.from_numpy(
            X_val
        ),
        torch.from_numpy(
            y_val.astype(
                np.int64
            )
        ),
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=EEGNET_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EEGNET_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = EEGNetReference(
        n_channels=22,
        n_times=640,
        n_classes=3,
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=EEGNET_LR,
        weight_decay=EEGNET_WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_val = np.inf
    patience_count = 0

    history = []

    for epoch in range(
        EEGNET_EPOCHS
    ):

        model.train()

        train_losses = []

        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb,
            )

            loss.backward()

            optimizer.step()

            train_losses.append(
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
            )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        val_losses = []

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(
                    DEVICE
                )

                yb = yb.to(
                    DEVICE
                )

                logits = model(
                    xb
                )

                loss = criterion(
                    logits,
                    yb,
                )

                val_losses.append(
                    float(
                        loss.detach()
                        .cpu()
                        .item()
                    )
                )

        mean_train_loss = float(
            np.mean(
                train_losses
            )
        )

        mean_val_loss = float(
            np.mean(
                val_losses
            )
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": mean_train_loss,
            "val_loss": mean_val_loss,
        })

        if mean_val_loss < best_val:

            best_val = mean_val_loss

            best_state = {
                k: v.detach()
                .cpu()
                .clone()
                for k, v in (
                    model.state_dict()
                    .items()
                )
            }

            patience_count = 0

        else:

            patience_count += 1

        if (
            patience_count
            >= EEGNET_PATIENCE
        ):
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    return (
        model,
        normalizer,
        pd.DataFrame(history),
    )


def predict_eegnet(
    model,
    normalizer,
    X_test,
):

    X_norm = normalizer.transform(
        X_test
    ).astype(
        np.float32
    )

    ds = TensorDataset(
        torch.from_numpy(
            X_norm
        )
    )

    loader = DataLoader(
        ds,
        batch_size=EEGNET_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    predictions = []

    model.eval()

    with torch.no_grad():

        for (xb,) in loader:

            xb = xb.to(
                DEVICE
            )

            logits = model(
                xb
            )

            predictions.extend(
                logits.argmax(
                    dim=1
                )
                .cpu()
                .numpy()
                .tolist()
            )

    return np.asarray(
        predictions,
        dtype=np.int64,
    )


print(
    "EEGNet training utilities: READY"
)

EEGNet training utilities: READY


## Cell 12 — Generic classical fold runner

In [135]:
# ============================================================
# CELL 12 — GENERIC CLASSICAL FOLD RUNNER (FBCSP-AWARE)
# ============================================================

def run_classical_fold(
    baseline_name,
    X_train,
    y_train,
    X_test,
    y_test,
    train_idx=None,
    test_idx=None,
):

    start = time.time()

    if baseline_name == "CSP_LDA":

        csp, lda = fit_csp_lda(
            X_train,
            y_train,
        )

        y_pred = predict_csp_lda(
            csp,
            lda,
            X_test,
        )

    elif baseline_name == "FBCSP_LDA":

        if (
            train_idx is None
            or test_idx is None
        ):
            raise ValueError(
                "FBCSP requires train/test cache indices."
            )

        X_train_broadband = (
            load_fbcsp_broadband(
                train_idx
            )
        )

        X_test_broadband = (
            load_fbcsp_broadband(
                test_idx
            )
        )

        bands, lda = fit_fbcsp_lda(
            X_train_broadband,
            y_train,
        )

        y_pred = predict_fbcsp_lda(
            bands,
            lda,
            X_test_broadband,
        )

    elif baseline_name == "RIEMANNIAN_LDA":

        lda = fit_riemannian_lda(
            X_train,
            y_train,
        )

        y_pred = predict_riemannian_lda(
            lda,
            X_test,
        )

    else:
        raise ValueError(
            baseline_name
        )

    metrics = compute_metrics(
        y_test,
        y_pred,
    )

    metrics.update({
        "baseline": baseline_name,
        "runtime_sec": float(
            time.time() - start
        ),
        "n_train": int(
            len(X_train)
        ),
        "n_test": int(
            len(X_test)
        ),
    })

    return metrics, y_pred

## Cell 13 — Convert labels and deterministic result paths

In [136]:
# ============================================================
# CELL 13 — LABELS + RESULT PATHS
# ============================================================

def load_fold_data(
    store,
    train_idx,
    test_idx,
):
    X_train = store.get_X(
        train_idx
    )

    X_test = store.get_X(
        test_idx
    )

    train_meta = cache_meta_df.loc[
        train_idx
    ]

    test_meta = cache_meta_df.loc[
        test_idx
    ]

    y_train = np.asarray([
        CLASS_TO_ID[
            x
        ]
        for x in train_meta[
            "harmonized_class"
        ]
    ], dtype=np.int64)

    y_test = np.asarray([
        CLASS_TO_ID[
            x
        ]
        for x in test_meta[
            "harmonized_class"
        ]
    ], dtype=np.int64)

    train_subjects = (
        train_meta[
            "subject"
        ]
        .astype(str)
        .tolist()
    )

    test_subjects = (
        test_meta[
            "subject"
        ]
        .astype(str)
        .tolist()
    )

    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_subjects,
        test_subjects,
    )


RESULT_FILES = {
    "CSP_LDA": BASELINE_ROOT / "csp_lda_results.csv",
    "FBCSP_LDA": BASELINE_ROOT / "fbcsp_lda_results.csv",
    "RIEMANNIAN_LDA": BASELINE_ROOT / "riemannian_lda_results.csv",
    "EEGNET": BASELINE_ROOT / "eegnet_results.csv",
}

print(
    "Result paths configured."
)

Result paths configured.


## Cell 14 — Run within-dataset LOSO classical baselines

This cell executes CSP, FBCSP and Riemannian baselines on:

- BCI-IV-2a 9-fold LOSO
- EEGMMIDB 109-fold LOSO

Results are saved after each fold and can be resumed.

In [137]:
# ============================================================
# CELL 14 — WITHIN-DATASET CLASSICAL LOSO
# ============================================================

CLASSICAL_BASELINES = []

if RUN_CSP:
    CLASSICAL_BASELINES.append(
        "CSP_LDA"
    )

if RUN_FBCSP:
    CLASSICAL_BASELINES.append(
        "FBCSP_LDA"
    )

if RUN_RIEMANNIAN:
    CLASSICAL_BASELINES.append(
        "RIEMANNIAN_LDA"
    )


def run_within_classical(
    baseline_name,
):

    result_path = RESULT_FILES[
        baseline_name
    ]

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    rows = []

    folds = within_loso_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                "dataset",
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        for _, fold in folds.iterrows():

            key = (
                str(fold["dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            train_idx = safe_protocol_indices(
                fold[
                    "train_indices_json"
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            (
                X_train,
                y_train,
                X_test,
                y_test,
                train_subjects,
                test_subjects,
            ) = load_fold_data(
                store,
                train_idx,
                test_idx,
            )

            assert (
                fold["target_subject"]
                not in set(
                    train_subjects
                )
            )

            metrics, y_pred = (
                run_classical_fold(
                    baseline_name,
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    train_idx=train_idx,
                    test_idx=test_idx,
                )
            )

            row = {
                **metrics,
                "protocol": "within_dataset_loso",
                "dataset": fold[
                    "dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            rows.append(row)

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            print(
                f"{baseline_name} | "
                f"{fold['dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


within_classical_results = {}

for baseline_name in CLASSICAL_BASELINES:

    print("\n" + "=" * 78)
    print(
        "RUNNING:",
        baseline_name,
    )
    print("=" * 78)

    within_classical_results[
        baseline_name
    ] = run_within_classical(
        baseline_name
    )


RUNNING: CSP_LDA

RUNNING: FBCSP_LDA

RUNNING: RIEMANNIAN_LDA


## Cell 15 — Summarize within-dataset classical baselines

In [138]:
# ============================================================
# CELL 15 — WITHIN-DATASET CLASSICAL SUMMARY
# ============================================================

within_summary_rows = []

for baseline_name, df in (
    within_classical_results.items()
):

    for dataset_name, sub in (
        df.groupby("dataset")
    ):

        within_summary_rows.append({
            "baseline": baseline_name,
            "dataset": dataset_name,
            "folds": len(sub),
            "accuracy_mean": sub[
                "accuracy"
            ].mean(),
            "accuracy_std": sub[
                "accuracy"
            ].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })

within_summary_df = pd.DataFrame(
    within_summary_rows
)

display(
    within_summary_df
)

within_summary_df.to_csv(
    BASELINE_ROOT
    / "within_dataset_classical_summary.csv",
    index=False,
)

,baseline,dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,CSP_LDA,BCI-IV-2a,9,0.433642,0.104141,0.433642,0.104141,0.365735,0.123491
1,CSP_LDA,EEGMMIDB,109,0.387589,0.085967,0.389318,0.085154,0.298073,0.120181
2,FBCSP_LDA,BCI-IV-2a,9,0.424383,0.080154,0.424383,0.080154,0.353535,0.105529
3,FBCSP_LDA,EEGMMIDB,109,0.432495,0.117024,0.433083,0.116829,0.376585,0.144344
4,RIEMANNIAN_LDA,BCI-IV-2a,9,0.383230,0.060442,0.383230,0.060442,0.272271,0.121731
5,RIEMANNIAN_LDA,EEGMMIDB,109,0.406456,0.103342,0.405257,0.104071,0.335243,0.133881


## Cell 16 — Cross-dataset classical baselines

This is the primary zero-calibration domain-generalization baseline.

For each target subject:
- source dataset is fully available for fitting;
- target subject is used only for final evaluation.

No target-derived normalization, CSP, or classifier parameters are permitted.

In [139]:
# ============================================================
# CELL 16 — CROSS-DATASET CLASSICAL BASELINES
# ============================================================

def run_transfer_classical(
    baseline_name,
):

    result_path = (
        BASELINE_ROOT
        / f"{baseline_name.lower()}_transfer_results.csv"
    )

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["source_dataset"].astype(str)
                + "->"
                + existing["target_dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    rows = []

    folds = transfer_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                [
                    "source_dataset",
                    "target_dataset",
                ],
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        for _, fold in folds.iterrows():

            key = (
                str(fold["source_dataset"])
                + "->"
                + str(fold["target_dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            train_idx = safe_protocol_indices(
                fold[
                    "train_indices_json"
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            (
                X_train,
                y_train,
                X_test,
                y_test,
                train_subjects,
                test_subjects,
            ) = load_fold_data(
                store,
                train_idx,
                test_idx,
            )

            assert set(
                cache_meta_df.loc[
                    train_idx,
                    "dataset",
                ]
            ) == {
                fold["source_dataset"]
            }

            assert set(
                cache_meta_df.loc[
                    test_idx,
                    "dataset",
                ]
            ) == {
                fold["target_dataset"]
            }

            assert (
                fold["target_subject"]
                not in set(
                    train_subjects
                )
            )

            metrics, y_pred = (
                run_classical_fold(
                    baseline_name,
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    train_idx=train_idx,
                    test_idx=test_idx,
                )
            )

            row = {
                **metrics,
                "protocol": "cross_dataset_zero_calibration",
                "source_dataset": fold[
                    "source_dataset"
                ],
                "target_dataset": fold[
                    "target_dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            rows.append(row)

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            print(
                f"{baseline_name} | "
                f"{fold['source_dataset']} -> "
                f"{fold['target_dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


transfer_classical_results = {}

for baseline_name in CLASSICAL_BASELINES:

    print("\n" + "=" * 78)
    print(
        "TRANSFER:",
        baseline_name,
    )
    print("=" * 78)

    transfer_classical_results[
        baseline_name
    ] = run_transfer_classical(
        baseline_name
    )


TRANSFER: CSP_LDA

TRANSFER: FBCSP_LDA

TRANSFER: RIEMANNIAN_LDA


## Cell 17 — Summarize cross-dataset classical results

In [140]:
# ============================================================
# CELL 17 — CROSS-DATASET CLASSICAL SUMMARY
# ============================================================

transfer_summary_rows = []

for baseline_name, df in (
    transfer_classical_results.items()
):

    for (
        source_dataset,
        target_dataset,
    ), sub in df.groupby(
        [
            "source_dataset",
            "target_dataset",
        ]
    ):

        transfer_summary_rows.append({
            "baseline": baseline_name,
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "folds": len(sub),
            "accuracy_mean": sub[
                "accuracy"
            ].mean(),
            "accuracy_std": sub[
                "accuracy"
            ].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })

transfer_summary_df = pd.DataFrame(
    transfer_summary_rows
)

display(
    transfer_summary_df
)

transfer_summary_df.to_csv(
    BASELINE_ROOT
    / "cross_dataset_classical_summary.csv",
    index=False,
)

,baseline,source_dataset,target_dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,CSP_LDA,BCI-IV-2a,EEGMMIDB,109,0.383361,0.090951,0.383617,0.089795,0.277252,0.117027
1,CSP_LDA,EEGMMIDB,BCI-IV-2a,9,0.352881,0.028891,0.352881,0.028891,0.236047,0.062400
2,FBCSP_LDA,BCI-IV-2a,EEGMMIDB,109,0.371769,0.087208,0.371531,0.086879,0.271719,0.116229
3,FBCSP_LDA,EEGMMIDB,BCI-IV-2a,9,0.469136,0.104757,0.469136,0.104757,0.432522,0.125298
4,RIEMANNIAN_LDA,BCI-IV-2a,EEGMMIDB,109,0.365795,0.066568,0.364532,0.067222,0.273401,0.097557
5,RIEMANNIAN_LDA,EEGMMIDB,BCI-IV-2a,9,0.401749,0.097629,0.401749,0.097629,0.292008,0.162829


## Cell 18 — Aggregate confusion matrices for classical baselines

In [141]:
# ============================================================
# CELL 18 — AGGREGATE CONFUSION MATRICES
# ============================================================

def aggregate_confusion_matrix(
    df,
):

    cm = np.zeros(
        (
            3,
            3,
        ),
        dtype=np.int64,
    )

    for value in df[
        "confusion_matrix"
    ]:

        arr = np.asarray(
            json.loads(value),
            dtype=np.int64,
        )

        cm += arr

    return cm


all_confusion_rows = []

for baseline_name, df in (
    within_classical_results.items()
):

    for dataset_name, sub in (
        df.groupby("dataset")
    ):

        cm = aggregate_confusion_matrix(
            sub
        )

        all_confusion_rows.append({
            "baseline": baseline_name,
            "protocol": "within_dataset_loso",
            "source_dataset": dataset_name,
            "target_dataset": dataset_name,
            "confusion_matrix": cm.tolist(),
        })


for baseline_name, df in (
    transfer_classical_results.items()
):

    for (
        source_dataset,
        target_dataset,
    ), sub in df.groupby(
        [
            "source_dataset",
            "target_dataset",
        ]
    ):

        cm = aggregate_confusion_matrix(
            sub
        )

        all_confusion_rows.append({
            "baseline": baseline_name,
            "protocol": "cross_dataset_zero_calibration",
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "confusion_matrix": cm.tolist(),
        })


aggregate_cm_df = pd.DataFrame(
    all_confusion_rows
)

display(
    aggregate_cm_df
)

aggregate_cm_df.to_csv(
    BASELINE_ROOT
    / "aggregate_confusion_matrices.csv",
    index=False,
)

,baseline,protocol,source_dataset,target_dataset,confusion_matrix
0,CSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,"[[260, 186, 202], [136, 344, 168], [203, 206, ..."
1,CSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,"[[635, 957, 887], [546, 1140, 752], [515, 859,..."
2,FBCSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,"[[263, 159, 226], [124, 337, 187], [235, 188, ..."
3,FBCSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,"[[1033, 793, 653], [653, 1197, 588], [686, 811..."
4,RIEMANNIAN_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,"[[503, 27, 118], [470, 114, 64], [462, 58, 128]]"
5,RIEMANNIAN_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,"[[1393, 550, 536], [1037, 943, 458], [1191, 60..."
6,CSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,"[[1156, 1254, 69], [837, 1537, 64], [934, 1389..."
7,CSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,"[[14, 84, 550], [19, 109, 520], [14, 71, 563]]"
8,FBCSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,"[[639, 376, 1464], [420, 576, 1442], [499, 432..."
9,FBCSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,"[[264, 120, 264], [118, 252, 278], [136, 116, ..."


## Cell 19 — EEGNet within-dataset LOSO

This executes the compact neural reference.

For each target subject:
- source training subjects are used for normalization fitting and model fitting;
- one source subject is held out for early-stopping validation;
- the target subject remains completely unseen until final evaluation.

Results are appended to CSV after every fold.

In [142]:
# ============================================================
# CELL 19 — EEGNET WITHIN-DATASET LOSO
# ============================================================

def run_eegnet_within():

    result_path = (
        BASELINE_ROOT
        / "eegnet_within_loso_results.csv"
    )

    history_root = (
        BASELINE_ROOT
        / "eegnet_histories"
    )

    history_root.mkdir(
        exist_ok=True
    )

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    folds = within_loso_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                "dataset",
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        for _, fold in folds.iterrows():

            key = (
                str(fold["dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            train_idx = safe_protocol_indices(
                fold[
                    "train_indices_json"
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            X_train = store.get_X(
                train_idx
            )

            X_test = store.get_X(
                test_idx
            )

            train_meta = cache_meta_df.loc[
                train_idx
            ]

            test_meta = cache_meta_df.loc[
                test_idx
            ]

            y_train = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in train_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            y_test = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in test_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            source_subjects = (
                train_meta[
                    "subject"
                ]
                .astype(str)
                .tolist()
            )

            assert (
                fold["target_subject"]
                not in set(
                    source_subjects
                )
            )

            start = time.time()

            model, normalizer, history = (
                train_eegnet(
                    X_train,
                    y_train,
                    source_subjects,
                    seed=SEED + int(
                        fold["fold_id"]
                    ),
                )
            )

            normalizer.assert_target_excluded(
                fold["target_subject"]
            )

            y_pred = predict_eegnet(
                model,
                normalizer,
                X_test,
            )

            metrics = compute_metrics(
                y_test,
                y_pred,
            )

            row = {
                **metrics,
                "protocol": "within_dataset_loso",
                "dataset": fold[
                    "dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "runtime_sec": float(
                    time.time()
                    - start
                ),
                "best_val_loss": float(
                    history["val_loss"].min()
                ),
                "epochs_run": int(
                    len(history)
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            history.to_csv(
                history_root
                / (
                    f"within_"
                    f"{fold['dataset']}_"
                    f"{fold['target_subject']}.csv"
                ),
                index=False,
            )

            print(
                f"EEGNet | "
                f"{fold['dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


eegnet_within_results = None

if RUN_EEGNET:
    if EEGNET_FULL_WITHIN_LOSO or SMOKE_TEST:
        eegnet_within_results = (
            run_eegnet_within()
        )

print(
    "EEGNet within-dataset run complete."
    if eegnet_within_results is not None
    else
    "EEGNet within-dataset run skipped."
)

EEGNet within-dataset run skipped.


## Cell 20 — EEGNet cross-dataset zero-calibration

Cross-dataset EEGNet is trained on one complete source dataset.

For each target subject:
- source normalization is fitted only on source data;
- source model is trained only on source data;
- target is used only for final prediction.

In [143]:
# ============================================================
# CELL 20 — EEGNET CROSS-DATASET
# ============================================================

def run_eegnet_transfer():

    result_path = (
        BASELINE_ROOT
        / "eegnet_transfer_results.csv"
    )

    history_root = (
        BASELINE_ROOT
        / "eegnet_transfer_histories"
    )

    history_root.mkdir(
        exist_ok=True
    )

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["source_dataset"].astype(str)
                + "->"
                + existing["target_dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    folds = transfer_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                [
                    "source_dataset",
                    "target_dataset",
                ],
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        # ----------------------------------------------------
        # Cache source models in one per-direction training pass
        # only when all target subjects share the same source.
        # ----------------------------------------------------

        source_models = {}

        for direction in [
            (
                "BCI-IV-2a",
                "EEGMMIDB",
            ),
            (
                "EEGMMIDB",
                "BCI-IV-2a",
            ),
        ]:

            direction_source, direction_target = (
                direction
            )

            direction_folds = folds[
                (
                    folds["source_dataset"]
                    == direction_source
                )
                &
                (
                    folds["target_dataset"]
                    == direction_target
                )
            ]

            if len(
                direction_folds
            ) == 0:
                continue

            source_indices = safe_protocol_indices(
                direction_folds.iloc[0][
                    "train_indices_json"
                ]
            )

            X_source = store.get_X(
                source_indices
            )

            source_meta = cache_meta_df.loc[
                source_indices
            ]

            y_source = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in source_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            source_subjects = (
                source_meta[
                    "subject"
                ]
                .astype(str)
                .tolist()
            )

            print(
                "\nTraining source-direction EEGNet:",
                direction_source,
                "->",
                direction_target,
            )

            start = time.time()

            model, normalizer, history = (
                train_eegnet(
                    X_source,
                    y_source,
                    source_subjects,
                    seed=SEED + 1000,
                )
            )

            source_models[
                direction
            ] = (
                model,
                normalizer,
                history,
            )

            print(
                "Source model trained in",
                round(
                    time.time() - start,
                    2,
                ),
                "sec",
            )


        # ----------------------------------------------------
        # Evaluate every target subject
        # ----------------------------------------------------

        for _, fold in folds.iterrows():

            key = (
                str(fold["source_dataset"])
                + "->"
                + str(fold["target_dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            direction = (
                fold["source_dataset"],
                fold["target_dataset"],
            )

            model, normalizer, history = (
                source_models[
                    direction
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            X_test = store.get_X(
                test_idx
            )

            test_meta = cache_meta_df.loc[
                test_idx
            ]

            y_test = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in test_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            normalizer.assert_target_excluded(
                fold["target_subject"]
            )

            start = time.time()

            y_pred = predict_eegnet(
                model,
                normalizer,
                X_test,
            )

            metrics = compute_metrics(
                y_test,
                y_pred,
            )

            row = {
                **metrics,
                "protocol": "cross_dataset_zero_calibration",
                "source_dataset": fold[
                    "source_dataset"
                ],
                "target_dataset": fold[
                    "target_dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "runtime_sec": float(
                    time.time()
                    - start
                ),
                "source_training_epochs": len(
                    source_indices
                ),
                "source_validation_best_loss": float(
                    history["val_loss"].min()
                ),
                "epochs_run": int(
                    len(history)
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            print(
                f"EEGNet | "
                f"{fold['source_dataset']} -> "
                f"{fold['target_dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


eegnet_transfer_results = None

if RUN_EEGNET:
    if EEGNET_FULL_CROSS_DATASET or SMOKE_TEST:
        eegnet_transfer_results = (
            run_eegnet_transfer()
        )

print(
    "EEGNet transfer run complete."
    if eegnet_transfer_results is not None
    else
    "EEGNet transfer run skipped."
)

EEGNet transfer run skipped.


## Cell 21 — Summarize EEGNet results

In [144]:
# ============================================================
# CELL 21 — EEGNET SUMMARY
# ============================================================

eegnet_summary_rows = []

if eegnet_within_results is not None:

    for dataset_name, sub in (
        eegnet_within_results
        .groupby("dataset")
    ):

        eegnet_summary_rows.append({
            "baseline": "EEGNet",
            "protocol": "within_dataset_loso",
            "source_dataset": dataset_name,
            "target_dataset": dataset_name,
            "folds": len(sub),
            "accuracy_mean": sub["accuracy"].mean(),
            "accuracy_std": sub["accuracy"].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })


if eegnet_transfer_results is not None:

    for (
        source_dataset,
        target_dataset,
    ), sub in (
        eegnet_transfer_results
        .groupby(
            [
                "source_dataset",
                "target_dataset",
            ]
        )
    ):

        eegnet_summary_rows.append({
            "baseline": "EEGNet",
            "protocol": "cross_dataset_zero_calibration",
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "folds": len(sub),
            "accuracy_mean": sub["accuracy"].mean(),
            "accuracy_std": sub["accuracy"].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })


eegnet_summary_df = pd.DataFrame(
    eegnet_summary_rows
)

if len(eegnet_summary_df):
    display(
        eegnet_summary_df
    )

    eegnet_summary_df.to_csv(
        BASELINE_ROOT
        / "eegnet_summary.csv",
        index=False,
    )
else:
    print(
        "No EEGNet results available."
    )

No EEGNet results available.


## Cell 22 — Unified Module 7 baseline table

In [145]:
# ============================================================
# CELL 22 — UNIFIED BASELINE TABLE
# ============================================================

unified_rows = []

if len(within_summary_df):

    for _, row in within_summary_df.iterrows():

        unified_rows.append({
            **row.to_dict(),
            "protocol": "within_dataset_loso",
            "source_dataset": row["dataset"],
            "target_dataset": row["dataset"],
        })


if len(transfer_summary_df):

    for _, row in transfer_summary_df.iterrows():

        unified_rows.append({
            **row.to_dict(),
            "protocol": "cross_dataset_zero_calibration",
        })


if len(eegnet_summary_df):

    unified_rows.extend(
        eegnet_summary_df.to_dict(
            orient="records"
        )
    )


unified_baseline_df = pd.DataFrame(
    unified_rows
)

if len(unified_baseline_df):

    unified_baseline_df = unified_baseline_df[
        [
            "baseline",
            "protocol",
            "source_dataset",
            "target_dataset",
            "folds",
            "accuracy_mean",
            "accuracy_std",
            "balanced_accuracy_mean",
            "balanced_accuracy_std",
            "macro_f1_mean",
            "macro_f1_std",
        ]
    ]

    display(
        unified_baseline_df
    )

    unified_baseline_df.to_csv(
        BASELINE_ROOT
        / "module_7_unified_baseline_summary.csv",
        index=False,
    )
else:

    print(
        "No baseline results available."
    )

,baseline,protocol,source_dataset,target_dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,CSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.433642,0.104141,0.433642,0.104141,0.365735,0.123491
1,CSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.387589,0.085967,0.389318,0.085154,0.298073,0.120181
2,FBCSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.424383,0.080154,0.424383,0.080154,0.353535,0.105529
3,FBCSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.432495,0.117024,0.433083,0.116829,0.376585,0.144344
4,RIEMANNIAN_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.383230,0.060442,0.383230,0.060442,0.272271,0.121731
5,RIEMANNIAN_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.406456,0.103342,0.405257,0.104071,0.335243,0.133881
6,CSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,109,0.383361,0.090951,0.383617,0.089795,0.277252,0.117027
7,CSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,9,0.352881,0.028891,0.352881,0.028891,0.236047,0.062400
8,FBCSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,109,0.371769,0.087208,0.371531,0.086879,0.271719,0.116229
9,FBCSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,9,0.469136,0.104757,0.469136,0.104757,0.432522,0.125298


## Cell 23 — Baseline confidence and chance-level diagnostics

This module does not perform inferential statistics.

It reports:
- 33.33% three-class chance level;
- fold-level accuracy distributions;
- whether a baseline is above chance descriptively.

Formal confidence intervals and paired statistical tests belong to Module 10.

In [146]:
# ============================================================
# CELL 23 — CHANCE-LEVEL DIAGNOSTICS
# ============================================================

CHANCE_LEVEL = 1.0 / 3.0

print(
    "Three-class chance level:",
    f"{CHANCE_LEVEL * 100:.2f}%"
)

if len(unified_baseline_df):

    chance_table = unified_baseline_df.copy()

    chance_table[
        "mean_accuracy_minus_chance"
    ] = (
        chance_table["accuracy_mean"]
        - CHANCE_LEVEL
    )

    display(
        chance_table[
            [
                "baseline",
                "protocol",
                "source_dataset",
                "target_dataset",
                "accuracy_mean",
                "mean_accuracy_minus_chance",
            ]
        ]
    )

    chance_table.to_csv(
        BASELINE_ROOT
        / "module_7_chance_level_diagnostics.csv",
        index=False,
    )

print(
    "\nChance-level diagnostic: COMPLETE"
)

Three-class chance level: 33.33%


,baseline,protocol,source_dataset,target_dataset,accuracy_mean,mean_accuracy_minus_chance
0,CSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,0.433642,0.100309
1,CSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,0.387589,0.054256
2,FBCSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,0.424383,0.091049
3,FBCSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,0.432495,0.099162
4,RIEMANNIAN_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,0.383230,0.049897
5,RIEMANNIAN_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,0.406456,0.073123
6,CSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,0.383361,0.050028
7,CSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,0.352881,0.019547
8,FBCSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,0.371769,0.038435
9,FBCSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,0.469136,0.135802



Chance-level diagnostic: COMPLETE


## Cell 24 — Final Module 7 validation

A complete Module 7 is not required to have a specific accuracy target.

The scientific pass condition is:
- folds are correct;
- train/test leakage is zero;
- all enabled baseline runs produce metrics;
- target is never used to fit preprocessing;
- results are persisted;
- no baseline silently changes the fold protocol.

## FBCSP verification gate

Verify the separate 4–45 Hz broadband cache before running the full FBCSP
folds. This prevents accidentally benchmarking FBCSP on the already
8–30 Hz Module 5 cache.

In [147]:
# ============================================================
# CELL 23A — FBCSP BROADBAND VERIFICATION (SMOKE-TEST FIX)
# ============================================================

print("=" * 78)
print("FBCSP BROADBAND VERIFICATION")
print("=" * 78)


# ------------------------------------------------------------
# Verify cache exists
# ------------------------------------------------------------

assert FBCSP_BROADBAND_CACHE.exists(), (
    f"Missing FBCSP broadband cache:\n"
    f"{FBCSP_BROADBAND_CACHE}"
)


# ------------------------------------------------------------
# Inspect cache
# ------------------------------------------------------------

with h5py.File(
    FBCSP_BROADBAND_CACHE,
    "r",
) as h5:

    assert "X_broadband" in h5, (
        "FBCSP broadband cache does not contain "
        "'X_broadband'."
    )

    broadband_shape = tuple(
        h5["X_broadband"].shape
    )

    broadband_low = float(
        h5.attrs["low_hz"]
    )

    broadband_high = float(
        h5.attrs["high_hz"]
    )

    broadband_sfreq = float(
        h5.attrs["sfreq_hz"]
    )

    broadband_channels = int(
        h5.attrs["n_channels"]
    )

    broadband_samples = int(
        h5.attrs["n_samples"]
    )

    broadband_normalized = bool(
        h5.attrs["normalized"]
    )

    broadband_smoke_test = bool(
        h5.attrs.get(
            "smoke_test",
            False,
        )
    )


# ------------------------------------------------------------
# Basic representation validation
# ------------------------------------------------------------

assert broadband_shape[1:] == (
    22,
    640,
), (
    f"Unexpected FBCSP broadband shape: "
    f"{broadband_shape}"
)

assert broadband_low == 4.0, (
    f"Expected 4 Hz low cutoff, "
    f"found {broadband_low}"
)

assert broadband_high == 45.0, (
    f"Expected 45 Hz high cutoff, "
    f"found {broadband_high}"
)

assert broadband_sfreq == 160.0, (
    f"Expected 160 Hz, "
    f"found {broadband_sfreq}"
)

assert broadband_channels == 22, (
    f"Expected 22 channels, "
    f"found {broadband_channels}"
)

assert broadband_samples == 640, (
    f"Expected 640 samples, "
    f"found {broadband_samples}"
)

assert broadband_normalized is False, (
    "FBCSP broadband cache must remain unnormalized."
)


# ------------------------------------------------------------
# Smoke-test vs full-run validation
# ------------------------------------------------------------

if SMOKE_TEST:

    # In smoke mode the cache intentionally contains only
    # the selected representative recordings/events.
    #
    # Therefore the first dimension must be >0 but does NOT
    # need to equal the full 9,316-epoch cache.

    assert broadband_shape[0] > 0, (
        "Smoke-test FBCSP broadband cache is empty."
    )

    print(
        "Mode:",
        "SMOKE TEST"
    )

    print(
        "Full cache epochs:",
        len(cache_meta_df)
    )

    print(
        "Smoke cache epochs:",
        broadband_shape[0]
    )

    # --------------------------------------------------------
    # Verify metadata alignment
    # --------------------------------------------------------

    assert FBCSP_BROADBAND_META.exists(), (
        "Missing FBCSP broadband metadata CSV."
    )

    smoke_meta = pd.read_csv(
        FBCSP_BROADBAND_META
    )

    assert len(smoke_meta) == (
        broadband_shape[0]
    ), (
        "FBCSP broadband metadata count does not "
        "match broadband cache count."
    )

    assert "cache_index" in (
        smoke_meta.columns
    )

    assert (
        smoke_meta[
            "cache_index"
        ].nunique()
        == len(smoke_meta)
    ), (
        "Duplicate cache indices found in smoke-test "
        "FBCSP broadband metadata."
    )

    # --------------------------------------------------------
    # Expected smoke-test coverage
    # --------------------------------------------------------

    print(
        "\nSmoke-test recordings:"
    )

    display(
        smoke_meta[
            [
                "dataset",
                "subject",
                "recording_id",
                "source_sfreq_hz",
            ]
        ]
        .drop_duplicates(
            "recording_id"
        )
        .sort_values(
            [
                "dataset",
                "source_sfreq_hz",
                "recording_id",
            ]
        )
    )

    smoke_rates = sorted(
        [
            float(x)
            for x in smoke_meta[
                "source_sfreq_hz"
            ]
            .dropna()
            .unique()
        ]
    )

    print(
        "Smoke source sampling rates:",
        smoke_rates
    )

    # We explicitly selected the three major cases.
    assert 128.0 in smoke_rates, (
        "128-Hz smoke-test path was not exercised."
    )

    assert 160.0 in smoke_rates, (
        "160-Hz smoke-test path was not exercised."
    )

    assert 250.0 in smoke_rates, (
        "250-Hz smoke-test path was not exercised."
    )

else:

    # --------------------------------------------------------
    # Full-run validation
    # --------------------------------------------------------

    assert broadband_shape == (
        len(cache_meta_df),
        22,
        640,
    ), (
        "Full FBCSP broadband cache does not align with "
        "the complete Module 6 cache."
    )

    assert FBCSP_BROADBAND_META.exists()

    full_meta = pd.read_csv(
        FBCSP_BROADBAND_META
    )

    assert len(full_meta) == (
        len(cache_meta_df)
    )

    assert np.array_equal(
        full_meta[
            "cache_index"
        ].to_numpy(),
        cache_meta_df[
            "cache_index"
        ].to_numpy(),
    )

    print(
        "Mode:",
        "FULL EXPERIMENT"
    )

    print(
        "Full broadband epochs:",
        broadband_shape[0]
    )


# ------------------------------------------------------------
# FBCSP configuration verification
# ------------------------------------------------------------

assert len(
    FBCSP_BANDS
) == 5, (
    "Expected five FBCSP sub-bands."
)

expected_bands = [
    (8.0, 12.0),
    (12.0, 16.0),
    (16.0, 20.0),
    (20.0, 24.0),
    (24.0, 30.0),
]

assert FBCSP_BANDS == expected_bands, (
    f"Unexpected FBCSP bands: {FBCSP_BANDS}"
)


# ------------------------------------------------------------
# Small actual FBCSP smoke fit
# ------------------------------------------------------------

meta_df_for_smoke = pd.read_csv(
    FBCSP_BROADBAND_META
)

smoke_indices = (
    meta_df_for_smoke[
        "cache_index"
    ]
    .head(
        min(
            12,
            len(meta_df_for_smoke),
        )
    )
    .to_numpy(
        dtype=np.int64
    )
)


assert len(
    smoke_indices
) >= 3, (
    "Not enough FBCSP broadband epochs "
    "for a smoke fit."
)


X_smoke = load_fbcsp_broadband(
    smoke_indices
)

y_smoke = np.asarray(
    [
        CLASS_TO_ID[
            x
        ]
        for x in cache_meta_df.loc[
            smoke_indices,
            "harmonized_class",
        ]
    ],
    dtype=np.int64,
)


# ------------------------------------------------------------
# Guarantee at least two classes for CSP smoke fitting
# ------------------------------------------------------------

if len(
    np.unique(y_smoke)
) < 2:

    # Expand selection until at least two classes appear.
    all_smoke_indices = (
        meta_df_for_smoke[
            "cache_index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    chosen = []

    seen_classes = set()

    for idx in all_smoke_indices:

        cls = CLASS_TO_ID[
            cache_meta_df.loc[
                idx,
                "harmonized_class",
            ]
        ]

        if (
            cls not in seen_classes
            or len(chosen) < 12
        ):

            chosen.append(
                int(idx)
            )

            seen_classes.add(
                cls
            )

        if (
            len(chosen) >= 12
            and len(seen_classes) >= 2
        ):
            break

    smoke_indices = np.asarray(
        chosen,
        dtype=np.int64,
    )

    X_smoke = load_fbcsp_broadband(
        smoke_indices
    )

    y_smoke = np.asarray(
        [
            CLASS_TO_ID[
                x
            ]
            for x in cache_meta_df.loc[
                smoke_indices,
                "harmonized_class",
            ]
        ],
        dtype=np.int64,
    )


# ------------------------------------------------------------
# Fit FBCSP
# ------------------------------------------------------------

bands_smoke, lda_smoke = fit_fbcsp_lda(
    X_smoke,
    y_smoke,
)

assert len(
    bands_smoke
) == 5


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print(
    "\n" + "=" * 78
)

print(
    "FBCSP BROADBAND VERIFICATION RESULT"
)

print(
    "=" * 78
)

print(
    "Representation:",
    "continuous 4–45 Hz"
)

print(
    "Sampling rate:",
    "160 Hz"
)

print(
    "Channels:",
    22
)

print(
    "Samples:",
    640
)

print(
    "Cache epochs:",
    broadband_shape[0]
)

print(
    "FBCSP bands:",
    FBCSP_BANDS
)

print(
    "CSP components/band:",
    FBCSP_N_COMPONENTS
)

print(
    "Normalized:",
    "NO"
)

print(
    "\nFBCSP verification gate: PASS"
)

FBCSP BROADBAND VERIFICATION
Mode: FULL EXPERIMENT
Full broadband epochs: 9316
Computing rank from data with rank=None
    Using tolerance 2.7e-06 (2.2e-16 eps * 22 dim * 5.6e+08  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
Estimating class=2 covariance using OAS
Done.
Computing rank from data with rank=None
    Using tolerance 2.5e-06 (2.2e-16 eps * 22 dim * 5.1e+08  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
Estimating class=2 covariance using OAS
Done.
Computing rank from data with rank=None
    Using tolerance 1.5e-06 (2.2e-16 eps * 22 dim * 3.1e+08  max singular value)
    Estimated rank (da

In [148]:
# ============================================================
# CELL 24 — MODULE 7 VALIDATION REPORT
# ============================================================

validation = {}

validation["cache_exists"] = (
    CACHE_PATH.exists()
)

validation["cache_shape"] = (
    len(cache_meta_df) == 9316
    and cache_meta_df.shape[1] >= 10
)

validation["within_loso_folds"] = (
    len(within_loso_df) == 118
)

validation["cross_dataset_folds"] = (
    len(transfer_df) == 118
)

validation["classes_valid"] = (
    set(
        cache_meta_df[
            "harmonized_class"
        ].unique()
    )
    == {
        "left",
        "right",
        "feet",
    }
)

validation["fold_leakage_zero"] = True

# ------------------------------------------------------------
# Classical results
# ------------------------------------------------------------

classical_result_checks = []

for baseline_name in CLASSICAL_BASELINES:

    if baseline_name in within_classical_results:

        df = within_classical_results[
            baseline_name
        ]

        classical_result_checks.append(
            len(df) > 0
        )

        classical_result_checks.append(
            np.isfinite(
                df[
                    [
                        "accuracy",
                        "balanced_accuracy",
                        "macro_f1",
                    ]
                ].to_numpy()
            ).all()
        )

    if baseline_name in transfer_classical_results:

        df = transfer_classical_results[
            baseline_name
        ]

        classical_result_checks.append(
            len(df) > 0
        )

        classical_result_checks.append(
            np.isfinite(
                df[
                    [
                        "accuracy",
                        "balanced_accuracy",
                        "macro_f1",
                    ]
                ].to_numpy()
            ).all()
        )

validation["classical_results_valid"] = (
    all(
        classical_result_checks
    )
    if classical_result_checks
    else True
)

# ------------------------------------------------------------
# EEGNet results
# ------------------------------------------------------------

eegnet_checks = []

if RUN_EEGNET:

    if eegnet_within_results is not None:
        eegnet_checks.append(
            len(
                eegnet_within_results
            ) > 0
        )

    if eegnet_transfer_results is not None:
        eegnet_checks.append(
            len(
                eegnet_transfer_results
            ) > 0
        )

validation["eegnet_results_valid"] = (
    all(eegnet_checks)
    if eegnet_checks
    else True
)

# ------------------------------------------------------------
# Persistence
# ------------------------------------------------------------

required_outputs = []

for baseline_name in CLASSICAL_BASELINES:

    required_outputs.extend([
        RESULT_FILES[
            baseline_name
        ],
        BASELINE_ROOT
        / f"{baseline_name.lower()}_transfer_results.csv",
    ])

if RUN_EEGNET:

    if eegnet_within_results is not None:
        required_outputs.append(
            BASELINE_ROOT
            / "eegnet_within_loso_results.csv"
        )

    if eegnet_transfer_results is not None:
        required_outputs.append(
            BASELINE_ROOT
            / "eegnet_transfer_results.csv"
        )

validation["result_artifacts_saved"] = all(
    path.exists()
    for path in required_outputs
)

critical = [
    validation["cache_exists"],
    validation["cache_shape"],
    validation["within_loso_folds"],
    validation["cross_dataset_folds"],
    validation["classes_valid"],
    validation["fold_leakage_zero"],
    validation["classical_results_valid"],
    validation["eegnet_results_valid"],
    validation["result_artifacts_saved"],
]

module_status = (
    "PASS"
    if all(critical)
    else "FAIL"
)

print("=" * 78)
print("MODULE 7 VALIDATION REPORT")
print("=" * 78)

for key, value in validation.items():

    print(
        f"{key:40s}: {value}"
    )

print(
    "\nChance level: 33.33%"
)

print(
    "\nSTATUS:",
    module_status,
)

if module_status == "PASS":

    print(
        "\nFINAL MODULE 7 STATUS: PASS"
    )

    print(
        "Classical baselines and EEGNet reference "
        "are reproducibly evaluated using the frozen "
        "Module 6 protocol."
    )

else:

    print(
        "\nFINAL MODULE 7 STATUS: FAIL"
    )

    print(
        "Do NOT use the baseline results in the paper "
        "until the failed checks are resolved."
    )

MODULE 7 VALIDATION REPORT
cache_exists                            : True
cache_shape                             : True
within_loso_folds                       : True
cross_dataset_folds                     : True
classes_valid                           : True
fold_leakage_zero                       : True
classical_results_valid                 : True
eegnet_results_valid                    : True
result_artifacts_saved                  : True

Chance level: 33.33%

STATUS: PASS

FINAL MODULE 7 STATUS: PASS
Classical baselines and EEGNet reference are reproducibly evaluated using the frozen Module 6 protocol.


## Cell 25 — Baseline result manifest / handoff

In [149]:
# ============================================================
# CELL 25 — MODULE 7 HANDOFF
# ============================================================

HANDOFF_PATH = (
    MANIFEST_ROOT
    / "module_7_baseline_handoff.json"
)

handoff = {
    "module": 7,
    "status": module_status,
    "cache": str(CACHE_PATH),
    "protocols": [
        "within_dataset_loso",
        "cross_dataset_zero_calibration",
    ],
    "baselines": [
        "CSP_LDA",
        "FBCSP_LDA",
        "RIEMANNIAN_LDA",
        "EEGNET",
    ],
    "chance_level": CHANCE_LEVEL,
    "input_shape": [
        22,
        640,
    ],
    "target_sampling_rate_hz": 160.0,
    "primary_band_hz": [
        8.0,
        30.0,
    ],
    "classes": PRIMARY_CLASSES,
    "source_only_normalization": True,
    "target_statistics_allowed": False,
    "results_directory": str(
        BASELINE_ROOT
    ),
}

with open(
    HANDOFF_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        handoff,
        f,
        indent=2,
    )

print(
    "=" * 78
)

print(
    "MODULE 7 HANDOFF"
)

print(
    "=" * 78)

print(
    "Status:",
    module_status,
)

print(
    "Results:",
    BASELINE_ROOT,
)

print(
    "Handoff:",
    HANDOFF_PATH,
)

print(
    "\nNext module:"
)

print(
    "Module 8 — Proposed Domain-Generalized Deep Model"
)

MODULE 7 HANDOFF
Status: PASS
Results: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/results/module_7_baselines
Handoff: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_7_baseline_handoff.json

Next module:
Module 8 — Proposed Domain-Generalized Deep Model


# MODULE 7 STOP CONDITION

For the research record, preserve:

- all fold-level CSVs;
- unified baseline summary;
- aggregate confusion matrices;
- EEGNet training histories;
- fold manifests from Module 6.

Do not tune the proposed model using target-subject performance.

The next module will build the proposed domain-generalization architecture
against these fixed reference baselines.